# Validation

In [1]:
import pandas as pd

In [7]:
# Paths
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
raw_path = f"{PATH}/data/raw/ibge_populacao/tabela6579.csv"
standardization_path = f"{PATH}/data/processed/ibge_populacao/ibge_populacao.parquet"

In [8]:
# Reading
raw = pd.read_csv(raw_path, sep=";", skiprows=3)
standardization = pd.read_parquet(standardization_path)

- Validation

In [9]:
resumo_raw_wide = pd.DataFrame({
    'tipo': raw.dtypes,
    'nulos': raw.isnull().sum(),
    'perc_nulos': (raw.isnull().sum() / len(raw)) * 100,
    'unicos': raw.nunique()
})

resumo_raw_wide

,tipo,nulos,perc_nulos,unicos
Cód.,object,1,0.017319,5761
Unidade da Federação e Município,object,169,2.926914,5605
2018,object,176,3.048147,5122
2019,object,176,3.048147,5104
2020,object,176,3.048147,5150
2021,object,176,3.048147,5137


In [10]:
resumo_standardization_long = pd.DataFrame({
    'tipo': standardization.dtypes,
    'nulos': standardization.isnull().sum(),
    'perc_nulos': (standardization.isnull().sum() / len(standardization)) * 100,
    'unicos': standardization.nunique()
})

resumo_standardization_long

,tipo,nulos,perc_nulos,unicos
municipio,object,676,2.926914,5324
uf,object,812,3.515760,27
cod_uf,object,0,0.000000,54
ano,object,0,0.000000,4
populacao,object,704,3.048147,16634
cod_municipio,object,124,0.536890,5731


### Destrinchando

1. Quantidade de linhas esperadas

In [12]:
num_municipios = raw.shape[0]
num_anos = 4

expected_rows = num_municipios * num_anos
actual_rows = standardization.shape[0]

print(expected_rows, actual_rows)

23096 23096


In [13]:
assert expected_rows == actual_rows

2. Verificando se todos os anos existem

In [18]:
anos_esperados = {'2018', '2019', '2020', '2021'}
anos_long = set(standardization["ano"].unique())

assert anos_esperados == anos_long

3. Validando duplicidade

In [19]:
duplicados = standardization.duplicated(
    subset=["cod_municipio", "ano"]
).sum()

print(duplicados)

168


In [20]:
assert duplicados == 0

AssertionError: 

In [23]:
# Filtra e exibe os registros duplicados
linhas_duplicadas = standardization[standardization.duplicated(subset=["cod_municipio", "ano"], keep=False)]

# Ordena para comparar visualmente
linhas_duplicadas.sort_values(by=["cod_municipio", "ano"]).head(100)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
5611,None,None,1,2018,None,"1 - Por determinação judicial, o Município de ..."
5616,None,None,1,2018,None,"1 - Por determinação judicial, o Município de ..."
11385,None,None,1,2019,None,"1 - Por determinação judicial, o Município de ..."
11390,None,None,1,2019,None,"1 - Por determinação judicial, o Município de ..."
17159,None,None,1,2020,None,"1 - Por determinação judicial, o Município de ..."
...,...,...,...,...,...,...
19,São Paulo,None,35,2018,45538936,None
20,Paraná,None,41,2018,11348937,None
21,Santa Catarina,None,42,2018,7075494,None
22,Rio Grande do Sul,None,43,2018,11329605,None


In [28]:
# Traz apenas as linhas onde o ano é igual a 2020
df_teste = linhas_duplicadas.loc[linhas_duplicadas['cod_uf'] == '35']

In [29]:
df_teste.head(100)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
19,São Paulo,None,35,2018,45538936,None
5793,São Paulo,None,35,2019,45919049,None
11567,São Paulo,None,35,2020,46289333,None
17341,São Paulo,None,35,2021,46649132,None


In [30]:
standardization.loc[standardization['cod_uf'] == '35'].head(100)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
19,São Paulo,None,35,2018,45538936,None
3294,Adamantina,SP,35,2018,35023,3500105
3295,Adolfo,SP,35,2018,3571,3500204
3296,Aguaí,SP,35,2018,35954,3500303
3297,Águas da Prata,SP,35,2018,8137,3500402
...,...,...,...,...,...,...
3388,Cabrália Paulista,SP,35,2018,4286,3508306
3389,Cabreúva,SP,35,2018,48966,3508405
3390,Caçapava,SP,35,2018,93488,3508504
3391,Cachoeira Paulista,SP,35,2018,33067,3508603


In [39]:
duplicados = (
    standardization
    .query("cod_municipio.notnull()", engine="python")
    .duplicated(subset=["cod_municipio", "ano"])
    .sum()
)

print(duplicados)

48


In [45]:
municipios = standardization.query(
    "cod_municipio.notnull()",
    engine="python"
)

duplicados_df = municipios[
    municipios.duplicated(
        subset=["cod_municipio", "ano"],
        keep=False
    )
]

duplicados_df.sort_values(
    ["cod_municipio", "ano"]
)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
5611,None,None,1,2018,None,"1 - Por determinação judicial, o Município de ..."
5616,None,None,1,2018,None,"1 - Por determinação judicial, o Município de ..."
11385,None,None,1,2019,None,"1 - Por determinação judicial, o Município de ..."
11390,None,None,1,2019,None,"1 - Por determinação judicial, o Município de ..."
17159,None,None,1,2020,None,"1 - Por determinação judicial, o Município de ..."
...,...,...,...,...,...,...
11427,None,None,8,2019,None,8 - População judicial do município Coronel Jo...
17185,None,None,8,2020,None,8 - População judicial do município Coronel Jo...
17201,None,None,8,2020,None,8 - População judicial do município Coronel Jo...
22959,None,None,8,2021,None,8 - População judicial do município Coronel Jo...


In [46]:
duplicados_df[
    ["municipio", "cod_municipio", "ano", "populacao"]
]

,municipio,cod_municipio,ano,populacao
5607,None,2 - Por determinação judicial e para efeito de...,2018,None
5611,None,"1 - Por determinação judicial, o Município de ...",2018,None
5612,None,2 - Por determinação judicial e para efeito de...,2018,None
5616,None,"1 - Por determinação judicial, o Município de ...",2018,None
5617,None,2 - Por determinação judicial e para efeito de...,2018,None
...,...,...,...,...
22975,None,8 - População judicial do município Coronel Jo...,2021,None
22981,None,2 - População judicial do município de Santa I...,2021,None
22982,None,3 - População judicial do município de Urucará...,2021,None
22983,None,4 - População judicial do município de Jacarea...,2021,None


In [49]:
pd.set_option('display.max_colwidth', None)

In [51]:
duplicados_df.loc[duplicados_df['ano'] == '2018'].head(1000)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
5607,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5611,None,None,1,2018,None,"1 - Por determinação judicial, o Município de Porto Velho-RO teve suspensos os resultados do CENSO IBGE 2015, devendo-se utilizar como válidos, até futura decisão em contrário, a população estimada para 2014, que foi de 494.013 habitantes. Processo Judicial nº 12316-40.2016.4.01.4100 - Seção Judiciária de Rondônia."
5612,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5616,None,None,1,2018,None,"1 - Por determinação judicial, o Município de Porto Velho-RO teve suspensos os resultados do CENSO IBGE 2015, devendo-se utilizar como válidos, até futura decisão em contrário, a população estimada para 2014, que foi de 494.013 habitantes. Processo Judicial nº 12316-40.2016.4.01.4100 - Seção Judiciária de Rondônia."
5617,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5631,None,None,2,2018,None,"2 - População judicial do município de Santa Isabel do Rio Negro-AM: entre 23.773 e 30.564 habitantes. Parecer de Força Executória nº 00007/2017/NUCOB-GEAC/PFAM/PGF/AGU, em trâmite na 3ª VF/AM."
5632,None,None,3,2018,None,"3 - População judicial do município de Urucará-AM: entre 16.981 e 23.772 habitantes. Parecer de Força Executória nº 00004/2017/NUCOB-GEAC/PFAM/PGF/AGU, em trâmite na 3ª VF/AM."
5633,None,None,4,2018,None,"4 - População judicial do município de Jacareacanga-PA: 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5634,None,None,5,2018,None,5 - População judicial do município de Paço do Lumiar - MA: superior a 156.216 habitantes. Processo Judicial nº13916-98.2017.4.01.3700 - Seção Judiciária do Maranhão- MA.
5637,None,None,8,2018,None,8 - População judicial do município Coronel João Sá-BA: 17.422 habitantes. Processo Judicial nº 0002222-53.2017.4.01.3306 - Vara Única de Paulo Afonso-BA.


In [52]:
duplicados_df.loc[duplicados_df['cod_municipio'] == '2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA.'].head(1000)

,municipio,uf,cod_uf,ano,populacao,cod_municipio
5607,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5612,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
5617,None,None,2,2018,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
11381,None,None,2,2019,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
11386,None,None,2,2019,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
11391,None,None,2,2019,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
17155,None,None,2,2020,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
17160,None,None,2,2020,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
17165,None,None,2,2020,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."
22929,None,None,2,2021,None,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA."


In [55]:
raw.loc[raw['Cód.'] == '2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA.'].head(1000)

,Cód.,Unidade da Federação e Município,2018,2019,2020,2021
5607,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA.",NaN,NaN,NaN,NaN,NaN
5612,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA.",NaN,NaN,NaN,NaN,NaN
5617,"2 - Por determinação judicial e para efeito de distribuição do Fundo de Participação dos Municípios, a população do Município de Jacareacanga-PA é de 41.487 habitantes. Processo Judicial nº 798-41.2011.4.01.3902, Seção Judiciária de Itaituba-PA.",NaN,NaN,NaN,NaN,NaN


In [58]:
# Mostra a quantidade de linhas para cada conteúdo único da coluna
print(raw.groupby('Cód.').size())

Cód.
-                                                                                                                                                                                                                                                         1
..                                                                                                                                                                                                                                                        1
...                                                                                                                                                                                                                                                       1
0                                                                                                                                                                                                                                              

In [60]:
# 1. Lista com os códigos de UF
codigos_uf_validos = ['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', 
                      '24', '25', '26', '27', '28', '29', '31', '32', '33', '35', 
                      '41', '42', '43', '50', '51', '52', '53']

# 2. Garantia de que a coluna seja tratada como string para a análise
codigos_como_string = standardization['cod_municipio'].astype(str).str.strip()

# 3. Definindo o que é um código de MUNICIPIO válido (6 ou 7 dígitos numéricos)
municipio_valido = (codigos_como_string.str.isnumeric()) & (codigos_como_string.str.len().isin([6, 7]))

# 4. Definindo o que é um código de UF válido
uf_valida = codigos_como_string.isin(codigos_uf_validos)

# 5. O dado é considerado "Geral Válido" se for um município válido OU uma UF válida
dados_validos = municipio_valido | uf_valida

# 6. Usando o til (~) para pegar a NEGAÇÃO (tudo que NÃO é válido)
erros_df = standardization.loc[~dados_validos]

# 7. Resultados
print(f"Quantidade de erros após remover as UFs: {len(erros_df)}")
print(erros_df['cod_municipio'].unique().tolist())

Quantidade de erros após remover as UFs: 812
[None, 'Fonte: IBGE - Estimativas de População', 'Notas', 'Em 2003:\nO contingente populacional estimado para o Município de Caraíbas, no Estado da Bahia foi reduzido em 5.006 habitantes, devido ao processo de sindicância recentemente encerrado que apurou sobre enumeração ocorrida quando da realização do Censo Demográfico 2000.\n\nEm 2004:', '1 - O contingente populacional estimado para o Município de Caraíbas, no Estado da Bahia foi reduzido em 5.006 habitantes, devido ao processo de sindicância recentemente encerrado que apurou sobre enumeração ocorrida quando da realização do Censo Demográfico 2000.', '2 - Inclusive 4 municípios com instalação prevista para 1º de janeiro de 2005: Aroeiras do Itaim (PI), Figueirão (MS), Ipiranga do Norte (MT) e Itanhangá (MT).\n\nEm 2008:\nInclusive o Município de Nazária - PI, com previsão de instalação em 1º de janeiro de 2009.\n\nEm 2009:\nPor determinação judicial e para efeito de distribuição do Fundo